# 06 — The MOS Capacitor
## The exact classical C-V curve vs. PADRE — and where both part company with real devices

The metal-oxide-semiconductor (MOS) capacitor is the gate of every MOSFET. Its capacitance-voltage (C-V) curve is the
standard way to measure oxide thickness, doping, flat-band voltage and oxide charge. It is also one of the few device
problems with an **exact** classical solution, which makes it an ideal benchmark for a simulator.

**Learning objectives**
1. Name the three operating regions (accumulation, depletion, inversion) and draw their band diagrams.
2. Compute $C_{ox}$, the flat-band voltage, the threshold voltage and $C_{min}$ from textbook formulas.
3. Implement the exact classical C-V curve and compare it point by point with PADRE.
4. Understand why high-frequency and low-frequency C-V differ, and how to compute a correct low-frequency curve.
5. Use the C-V curve as a measurement: extract oxide thickness, doping and oxide charge.

**Device** (factory default): n⁺ polysilicon gate, 2 nm SiO₂, 5 µm of p-type silicon doped $10^{16}$ cm⁻³, ohmic back
contact, 1 µm × 1 µm gate. (The factory mesh starts at x = 0.001 µm to match the nanoHUB reference deck, so the
simulated gate is 0.999 µm wide.)

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
# PADRE is a compiled simulator. On nanoHUB it lives in an environment
# module that must be loaded into this Python kernel before the first
# simulation runs; elsewhere, PADRE just needs to be on your PATH.
import shutil
from nanohubpadre import use

if shutil.which("padre") is None:
    try:
        use("padre-2.4E-r15")          # nanoHUB module name
    except Exception as exc:           # not on nanoHUB: that is fine
        print("Could not load the PADRE module:", exc)

print("PADRE executable:", shutil.which("padre") or "NOT FOUND - simulations will fail")

In [ ]:
# ── Physical constants: textbook values vs. the values PADRE really uses ──
# PADRE prints its constants with `MODELS ... PRINT`. A fair comparison of
# simulation against a formula must feed the formula PADRE's numbers:
# e.g. the textbook n_i = 1.5e10 cm^-3 alone changes every analytic diode
# current by (1.5e10 / 9.963e9)^2 = 2.3x before any physics is compared.
import numpy as np

Q      = 1.602e-19        # elementary charge (C)
KT     = 0.025851         # thermal voltage kT/q at 300 K (V)
EPS0   = 8.854e-14        # vacuum permittivity (F/cm)
EPS_SI = 11.8 * EPS0      # silicon permittivity in PADRE (F/cm)
EPS_OX = 3.9 * EPS0       # SiO2 permittivity (F/cm)
NI     = 9.963e9          # intrinsic carrier density of Si at 300 K (cm^-3)
EG     = 1.12             # band gap (eV)
CHI_SI = 4.17             # electron affinity of Si in PADRE (eV)
NC, NV = 3.2e19, 2.03e19  # effective densities of states (cm^-3)

# The values most textbooks quote (e.g. Sze & Ng, Pierret), for contrast.
TEXTBOOK = dict(NI=1.5e10, EPS_SI=11.7 * EPS0, CHI_SI=4.05, NC=2.8e19, NV=1.04e19)


def compare(rows, title="Textbook vs. PADRE"):
    """Print a comparison table.

    rows: list of (quantity, textbook_value, padre_value, unit, why_different)
    """
    print(title)
    print("-" * 125)
    print(f"{'quantity':<40}{'textbook':>12}{'PADRE':>12}{'diff':>9}   why they differ")
    print("-" * 125)
    for name, tb, pd, unit, why in rows:
        d = f"{(pd - tb) / tb * 100:>+8.1f}%" if tb else "     n/a "
        print(f"{name + ' (' + unit + ')':<40}{tb:>12.4g}{pd:>12.4g}{d}   {why}")
    print("-" * 125)


compare([
    ("intrinsic density n_i", TEXTBOOK["NI"], NI, "cm^-3",
     "modern value (Sproul & Green 1991); 1.5e10 is the older textbook number"),
    ("Si permittivity", TEXTBOOK["EPS_SI"], EPS_SI, "F/cm", "PADRE uses eps_r = 11.8, Sze uses 11.7"),
    ("electron affinity chi", TEXTBOOK["CHI_SI"], CHI_SI, "eV", "PADRE (like Silvaco Atlas) uses 4.17"),
    ("conduction-band DOS Nc", TEXTBOOK["NC"], NC, "cm^-3", "different effective-mass fit"),
    ("valence-band DOS Nv", TEXTBOOK["NV"], NV, "cm^-3", "different effective-mass fit"),
], title="Material constants")

---
## 1. Theory

### 1.1 Regions of operation (p-type substrate)
| Gate bias | Surface | Capacitance |
|---|---|---|
| $V_G < V_{FB}$ | holes pile up: **accumulation** | ≈ $C_{ox}$ |
| $V_{FB} < V_G < V_T$ | holes pushed away: **depletion** | $C_{ox}$ in series with $C_d=\varepsilon_s/W_d$, falls as $W_d$ grows |
| $V_G > V_T$ | electrons gather: **inversion** | LF: back to $C_{ox}$. HF: stays at $C_{min}$ |

### 1.2 Key formulas (Sze & Ng Ch. 4; Pierret Ch. 16)
$$C_{ox}=\frac{\varepsilon_{ox}}{t_{ox}},\qquad \phi_F=\frac{kT}{q}\ln\frac{N_A}{n_i},\qquad
V_{FB}=\phi_{ms}=\phi_m-\Big(\chi+\frac{E_g}{2}+\phi_F\Big),$$
$$V_T=V_{FB}+2\phi_F+\frac{\sqrt{2q\varepsilon_sN_A(2\phi_F)}}{C_{ox}},\qquad
W_m=\sqrt{\frac{4\varepsilon_s\phi_F}{qN_A}},\qquad C_{min}=\frac{\varepsilon_{ox}}{t_{ox}+(\varepsilon_{ox}/\varepsilon_s)W_m}.$$
An n⁺ polysilicon gate has its Fermi level at the conduction band, so $\phi_m=\chi$ and $V_{FB}=-(E_g/2+\phi_F)$.

### 1.3 The exact classical solution
The depletion approximation is only needed for closed forms. Integrating Poisson's equation *exactly* with Boltzmann
statistics (Kingston & Neustadter 1955; Sze & Ng Eq. 4.14) gives the surface charge as a function of surface potential
$\psi_s$ ($u=q\psi_s/kT$, Debye length $L_D=\sqrt{\varepsilon_skT/q^2N_A}$):
$$Q_s=\mp\frac{\sqrt2\,\varepsilon_s kT}{qL_D}F,\qquad F=\sqrt{\big(e^{-u}+u-1\big)+\frac{n_{p0}}{p_{p0}}\big(e^{u}-u-1\big)}.$$
Then $V_G=V_{FB}+\psi_s-Q_s/C_{ox}$, and the low-frequency capacitance is $C_{ox}$ in series with $C_s=dQ_s/d\psi_s$.
This is the curve a perfect 1-D simulator should reproduce.

In [ ]:
def mos_cv_exact(vg, Na, tox_um, vfb, area_cm2):
    # Exact classical LF and HF C-V (Boltzmann statistics, uniform doping). Returns C in farads.
    p0, n0 = Na, NI ** 2 / Na
    LD = np.sqrt(EPS_SI * KT / (Q * p0))
    cox = EPS_OX / (tox_um * 1e-4)
    psi = np.linspace(-0.45, 1.3, 60001)
    u = psi / KT
    F = np.sqrt(np.maximum((np.exp(-u) + u - 1) + n0 / p0 * (np.exp(u) - u - 1), 1e-30))
    Qs = -np.sign(psi) * np.sqrt(2) * EPS_SI * KT / LD * F
    Cs_lf = EPS_SI / (np.sqrt(2) * LD) * np.abs(1 - np.exp(-u) + n0 / p0 * (np.exp(u) - 1)) / F
    # HF: minority carriers cannot follow the AC signal, so the depletion edge sets C.
    # Textbook treatment: for psi_s beyond 2*phi_F the depletion width is frozen at W_m.
    phiF = KT * np.log(Na / NI)
    Fd = np.sqrt(np.maximum(np.exp(-u) + u - 1, 1e-30))
    Cs_dep = EPS_SI / (np.sqrt(2) * LD) * np.abs(1 - np.exp(-u)) / Fd
    Cs_hf = np.where(psi < 0, Cs_lf, Cs_dep)
    Cs_hf = np.where(psi > 2 * phiF, np.interp(2 * phiF, psi, Cs_dep), Cs_hf)
    small = np.abs(psi) < 1e-9
    Cs_lf[small] = Cs_hf[small] = EPS_SI / LD
    V = vfb + psi - Qs / cox
    o = np.argsort(V)
    c_lf = area_cm2 * cox * Cs_lf / (cox + Cs_lf)
    c_hf = area_cm2 * cox * Cs_hf / (cox + Cs_hf)
    return np.interp(vg, V[o], c_lf[o]), np.interp(vg, V[o], c_hf[o])

Na, tox = 1e16, 0.002
AREA = 0.999e-4 * 1e-4
COX = EPS_OX / (tox * 1e-4) * AREA
phiF = KT * np.log(Na / NI)
VFB = -(EG / 2 + phiF)
VT = VFB + 2 * phiF + np.sqrt(2 * Q * EPS_SI * Na * 2 * phiF) / (EPS_OX / (tox * 1e-4))
WM = np.sqrt(4 * EPS_SI * phiF / (Q * Na))
CMIN = EPS_OX / (tox * 1e-4 + EPS_OX / EPS_SI * WM) * AREA
print(f"Cox = {COX:.4e} F   phi_F = {phiF:.4f} V   V_FB = {VFB:.3f} V   V_T = {VT:.3f} V")
print(f"W_m = {WM * 1e4:.3f} um   C_min = {CMIN:.4e} F   C_min/Cox = {CMIN / COX:.4f}")

---
## 2. Simulating the C-V curve

The factory sweeps the gate from −2 V to +2 V twice: once with a 1 MHz small-signal (AC) analysis, the
**high-frequency** curve, and once at 1 Hz, the **low-frequency** curve. It ramps the gate to the start of each sweep
in small steps. You will see a warning about the 1 Hz sweep; section 4 explains it.

In [ ]:
from nanohubpadre import create_mos_capacitor
from nanohubpadre.parser import parse_ac_file
import os

sim = create_mos_capacitor(log_cv=True, log_cv_lf=True, vg_sweep=(-2.0, 2.0, 0.1))
print(sim.run().returncode)
hf = parse_ac_file(os.path.join(sim.working_dir, "cv_data"))
lf = parse_ac_file(os.path.join(sim.working_dir, "cv_lf_data"))
vg, c_hf = hf.get_cv_data(gate_electrode=1)            # farads (per um of depth)
vq, c_qs = hf.get_quasistatic_cv(gate_electrode=1)     # dQ/dV of the DC gate charge
c_lf_th, c_hf_th = mos_cv_exact(vg, Na, tox, VFB, AREA)
c_qs_th, _ = mos_cv_exact(vq, Na, tox, VFB, AREA)   # the LF (quasi-static) theory curve

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4.8))
plt.plot(vg, c_hf / COX, "o", ms=4, label="PADRE HF (1 MHz)")
plt.plot(vq, c_qs / COX, "s", ms=4, label="PADRE low-freq. (quasi-static)")
plt.plot(vg, c_hf_th / COX, "--", label="exact theory, HF")
plt.plot(vg, c_lf_th / COX, ":", label="exact theory, LF")
for x0, lab in ((VFB, "$V_{FB}$"), (VT, "$V_T$")):
    plt.axvline(x0, ls=":", color="gray"); plt.text(x0 + 0.03, 0.05, lab)
plt.xlabel("gate voltage (V)"); plt.ylabel("C / Cox")
plt.title("MOS capacitor C-V: simulation vs. exact classical theory")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
def at(v_arr, c_arr, v0):
    return c_arr[np.argmin(abs(v_arr - v0))]

compare([
    ("C accumulation (-2 V)", at(vg, c_lf_th, -2.0), at(vg, c_hf, -2.0), "F", "exact theory: accumulation layer adds series C"),
    ("C accumulation vs Cox", COX, at(vg, c_hf, -2.0), "F", "Cox ignores the accumulation-layer capacitance"),
    ("C at flat band", at(vg, c_hf_th, VFB), at(vg, c_hf, VFB), "F", "C_FB = Cox || eps_s/L_D"),
    ("C_min (HF, +2 V)", CMIN, at(vg, c_hf, 2.0), "F", "psi_s rises past 2*phi_F in strong inversion"),
    ("C low-freq. (+2 V)", at(vq, c_qs_th, 2.0), at(vq, c_qs, 2.0), "F", "inversion layer is classical in both"),
], title="C-V: exact classical theory vs PADRE")

**What is different, and why**
* **Accumulation is not quite $C_{ox}$.** The hole accumulation layer has a finite thickness, a few Debye lengths
  in classical theory, which acts as a capacitor in series with the oxide. With a 2 nm oxide this lowers $C$ by ~5 %,
  and the exact theory and PADRE agree on it.
* **Depletion:** theory and simulation lie on top of each other. This is the region the depletion approximation
  handles well.
* **$C_{min}$ is ~3 % below the textbook formula.** The textbook freezes the surface potential at $2\phi_F$ at
  threshold. In strong inversion $\psi_s$ keeps creeping up by a few $kT/q$ (logarithmically with gate charge), so
  $W_d$ ends slightly wider than $W_m$ and $C$ slightly lower. PADRE solves the full problem; the formula is an approximation.
* **Differences from real devices that *neither* model includes.** At 2 nm, real accumulation and inversion capacitance
  is reduced a further 10–20 % by (i) the **quantum-mechanical** displacement of the carrier distribution from the
  interface (the "dark space", ≈1 nm), and (ii) **polysilicon depletion** in the gate. Both are well-known reasons
  why measured "electrical" oxide thickness exceeds the physical thickness.

---
## 3. The low-frequency curve and a numerical-method lesson

In a real LF measurement the gate voltage changes so slowly that minority carriers (electrons) are generated fast enough
to follow. Capacitance then returns to $C_{ox}$ in inversion. There are two ways to get this from a simulator:

1. **Small-signal AC at low frequency** (1 Hz): solve the linearised equations at each bias.
2. **Quasi-static**: take the DC gate charge $Q_G(V_G)$ from the steady-state solutions and differentiate, $C=dQ_G/dV_G$.
   This is the *definition* of the equilibrium LF capacitance.

PADRE's AC solve is accurate at MHz frequencies but **breaks down at the very low frequencies** where inversion responds.
Physics requires that the rows of the capacitance matrix sum to zero, $C_{11}+C_{12}=0$ (raising both terminals together
changes nothing). The 1 Hz data violates this badly:

In [ ]:
err_hf = hf.conservation_error()
err_lf = lf.conservation_error()
vl, c_lf_ac = lf.get_cv_data(gate_electrode=1)
for v0 in (-2.0, -1.0, 0.0, 0.5, 1.0, 2.0):
    k = np.argmin(abs(vl - v0))
    print(f"Vg = {v0:+.1f} V   HF: |C11+C12|/|C11| = {err_hf[k]:8.1e}    1 Hz AC: {err_lf[k]:8.1e}"
          f"   (1 Hz C/Cox = {c_lf_ac[k] / COX:5.2f}, quasi-static C/Cox = {at(vq, c_qs, v0) / COX:5.2f})")

**What is different, and why:** the 1 MHz solution conserves charge to 10⁻⁷, so it is trustworthy. The 1 Hz AC
solution is fine in accumulation and depletion, but in inversion its conservation error reaches tens to thousands
of percent. Its "capacitance" there, sometimes above $C_{ox}$, which is physically impossible, is numerical noise. The
**quasi-static** curve (`get_quasistatic_cv()`) uses only converged DC solutions and matches exact theory to ~1 %.

> **General lesson.** A simulator can be excellent in one mode (DC, MHz AC) and unreliable in another (sub-Hz AC). Always test
> a result against an exact identity such as charge conservation, current continuity or reciprocity before believing it.

---
## 4. Using C-V as a measurement

### 4.1 Oxide thickness
In strong accumulation, $C\approx C_{ox}$, so $t_{ox}\approx\varepsilon_{ox}A/C_{acc}$. Because of the accumulation-layer
capacitance, this "electrical" thickness comes out *thicker* than the physical one:

In [ ]:
rows = []
for tox_i in (0.002, 0.005, 0.010):
    s = create_mos_capacitor(oxide_thickness=tox_i, log_cv=True, vg_sweep=(-3.0, 2.0, 0.25))
    s.run()
    d = parse_ac_file(os.path.join(s.working_dir, "cv_data"))
    v_i, c_i = d.get_cv_data(gate_electrode=1)
    t_el = EPS_OX * AREA / c_i[0] * 1e7          # nm
    rows.append((f"t_ox from C_acc, tox = {tox_i * 1e3:.0f} nm", tox_i * 1e3, t_el, "nm",
                 "accumulation-layer C in series (classical dark space)"))
compare(rows, title="Electrical vs physical oxide thickness")

The electrical thickness comes out ~2–2.6 % larger than the physical one. The error is not a fixed offset: at
a fixed gate voltage (−3 V here) a thicker oxide induces less accumulation charge, so its accumulation layer is
wider and adds proportionally more. The "measured" oxide thickness therefore depends on how hard the surface is
accumulated. In real devices the quantum dark space and poly depletion add several tenths of a nm more.

### 4.2 Doping from $C_{min}$
A higher doping gives a narrower maximum depletion width, so a higher $C_{min}$. Inverting the $C_{min}$ formula
recovers $N_A$:

In [ ]:
from scipy.optimize import brentq
rows = []
for Na_i in (1e16, 1e17):
    s = create_mos_capacitor(oxide_thickness=0.005, substrate_doping=Na_i, log_cv=True, vg_sweep=(-2.0, 3.0, 0.25))
    s.run()
    d = parse_ac_file(os.path.join(s.working_dir, "cv_data"))
    v_i, c_i = d.get_cv_data(gate_electrode=1)
    cmin = c_i.min()
    cox5 = EPS_OX / 5e-7 * AREA
    def f(logN):
        N = 10 ** logN
        wm = np.sqrt(4 * EPS_SI * KT * np.log(N / NI) / (Q * N))
        return EPS_OX / (5e-7 + EPS_OX / EPS_SI * wm) * AREA - cmin
    rows.append((f"N_A from C_min (true {Na_i:.0e})", Na_i, 10 ** brentq(f, 14, 19), "cm^-3",
                 "textbook C_min formula slightly underestimates C_min"))
compare(rows, title="Substrate doping extracted from C_min")

**What is different, and why:** the extracted doping comes out about 7 % **low**. The true $C_{min}$ is ~3 % below the
textbook formula (§2), and the formula can only produce a lower $C_{min}$ with a lower $N_A$. This is a genuine
systematic error of the textbook extraction method, and experimentalists correct for it.

### 4.3 Oxide charge shifts the flat-band voltage, and a simulator surprise
A sheet of fixed positive charge $Q_{it}$ (cm⁻²) at the Si/SiO₂ interface shifts the curve by exactly
$\Delta V_{FB}=-qQ_{it}/C_{ox}$, to the left. The factory passes `oxide_qftrap` to PADRE's `INTERFACE QF`, which the
PADRE manual defines as a "fixed charge density (/cm²)". With a 10 nm oxide the shift is large enough to see clearly.

In [ ]:
curves = {}
for qit in (0.0, 5e11, 1e12):
    s = create_mos_capacitor(oxide_thickness=0.010, oxide_qftrap=qit, log_cv=True, vg_sweep=(-3.0, 2.0, 0.1))
    s.run()
    d = parse_ac_file(os.path.join(s.working_dir, "cv_data"))
    curves[qit] = d.get_cv_data(gate_electrode=1)

cox10 = EPS_OX / 10e-7                 # F/cm^2
v0, c0 = curves[0.0]
target = 0.5 * (c0.max() + c0.min())   # track the mid-transition point
def v_at(v, c):
    o = np.argsort(c)
    return np.interp(target, c[o], v[o])
rows = []
for qit, (v_i, c_i) in curves.items():
    if qit:
        rows.append((f"dV_FB for Q_it = {qit:.0e} cm^-2", -Q * qit / cox10, v_at(v_i, c_i) - v_at(v0, c0), "V",
                     "PADRE 2.4E: shift = +2 qQ/Cox (see below)"))
compare(rows, title="Flat-band shift from interface charge (t_ox = 10 nm)")

plt.figure(figsize=(7, 4))
for qit, (v_i, c_i) in curves.items():
    plt.plot(v_i, c_i / c_i.max(), label=f"oxide_qftrap = {qit:.0e} cm$^{{-2}}$")
plt.xlabel("$V_G$ (V)"); plt.ylabel("C / C_acc")
plt.title("PADRE: QF shifts the HF C-V curve to the right, by 2qQ/Cox")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**What is different, and why.** The textbook shift is exact: Gauss's law, no approximation. PADRE's shift is
**linear in the charge, but has the opposite sign and twice the size**: $+2qQ/C_{ox}$. We confirmed this with both
signs of $Q$ and by reading the shift independently of the curve-tracking method. The manual does not explain it,
so we report it as measured rather than guess at PADRE's internal convention. **Practical rule:** with PADRE 2.4E, to
model a physical positive interface charge $Q$, set `oxide_qftrap = -Q/2`.

This is the kind of discovery that only a comparison with an exact result can make. The C-V curve *looks* perfectly
reasonable either way.

---
## 5. Band diagrams in the three regions

In [ ]:
for vend, label in ((-1.5, "accumulation"), (-0.6, "depletion"), (1.5, "inversion")):
    s = create_mos_capacitor(log_bands_eq=False, log_bands_bias=True, log_cv=True,
                             vg_sweep=(0.0, vend, 0.1 if vend > 0 else -0.1))
    s.run()
    s.plot_band_diagram(title=f"Vg = {vend} V ({label}); gate oxide on the left, 0-2 nm")

* **Accumulation:** bands bend *up* at the surface, and $E_v$ approaches $E_F$ (holes gather).
* **Depletion:** bands bend *down* over ~0.2–0.3 µm, and the surface is depleted of holes.
* **Inversion:** bands bend down by about $2\phi_F$ = 0.71 V. At the surface $E_c$ comes closer to $E_F$ than $E_v$ does
  in the bulk: the surface has become n-type.

---
## 6. Summary: textbook vs. simulation for the MOS capacitor

| Quantity | Textbook | PADRE | What differs |
|---|---|---|---|
| $C$ in accumulation | $C_{ox}$ | ~5 % below $C_{ox}$ (2 nm) | finite accumulation-layer thickness (exact theory agrees with PADRE) |
| Depletion C-V | depletion approx. | matches exact theory ~1 % | nothing |
| $C_{min}$ | $W_m$ at $\psi_s=2\phi_F$ | ~3 % lower | $\psi_s$ exceeds $2\phi_F$ in strong inversion |
| LF C-V | returns to $C_{ox}$ | quasi-static: ~1 %; 1 Hz AC: noise | the simulator's AC solve fails at very low frequency |
| $\Delta V_{FB}$ from `oxide_qftrap` | $-qQ/C_{ox}$ | $+2qQ/C_{ox}$ | PADRE 2.4E's `INTERFACE QF` convention (measured) |
| Real 2 nm devices | — | — | quantum dark space and poly depletion, in *neither* model |

---
## 7. Exercises
1. **n-type substrate.** Use `substrate_type="n"` with a p⁺ poly gate (`gate_type="p_poly"`). Predict $V_{FB}$ and the
   direction of the C-V curve, then check.
2. **Metal gate.** With `gate_type="metal", gate_workfunction=4.6`, predict the shift of the curve relative to n⁺
   poly ($\Delta V_{FB}=\Delta\phi_m$). Remember PADRE's $\chi$ = 4.17 eV.
3. **Thin body.** Set `silicon_thickness=0.1` with $N_A$ = 1e17. Why does the factory warn? What happens to $C_{min}$
   when the back contact sits inside the would-be depletion region?
4. **Double gate.** Try `gate_config="double"` with a 20 nm body. Why is there no clear depletion "dip"?

**Next →** [04 — MOSFET](04_MOSFET.ipynb)